# Построение всех 6 LanceDB-баз (Colab)

Один прогон → 6 таблиц на Drive: для трёх моделей (rosberta, e5-base, bge-m3) ×
двух вариантов (base / fine-tuned).

**Ключевая гарантия:** во всех 6 таблицах ровно одни и те же 50 000 постов
(одни и те же `post_id`, в одном и том же порядке). Семплинг идёт один раз,
до загрузки моделей, с фиксированным `RANDOM_SEED = 42`. После этого тот же
список `posts` переиспользуется для каждой модели — меняется только вектор.

## Что нужно перед запуском

**1. GPU-рантайм:** Runtime → Change runtime type → T4 GPU (или лучше).

**2. Структура на Drive:**

```
Google Drive/
└── thesis/
    ├── data/
    │   └── posts/ajtkulov/selected/selected500k_cleaned.jsonl
    └── models/
        ├── bi-encoder-rosberta-finetuned.tar.gz   ← если есть
        ├── bi-encoder-e5-finetuned.tar.gz
        └── bi-encoder-bge-m3-finetuned.tar.gz
```

Архивы fine-tuned моделей ноутбук распакует сам в `/content/models/` —
это быстрее, чем читать веса прямо с Drive.

**3. В секции MODELS ниже** включи/отключи нужные пресеты, проставив
`enabled: True/False`. По умолчанию включены все 6.

**4. LanceDB-версия** должна совпадать с твоей локальной (`LANCEDB_VERSION`).

Результат: на Drive в `thesis/data/lancedb/` лежат 6 архивов
`<table_name>_lance.tar.gz`. Скачиваешь, распаковываешь в `lancedb_store/`,
готово.


In [ ]:
# Установка зависимостей, удаление конфликтного triton, монтирование Drive
LANCEDB_VERSION = "0.21.1"   # <-- поставь свою локальную версию

# ВАЖНО: pip install ПЕРЕД import torch, чтобы не было конфликта triton
!pip install -q --upgrade-strategy only-if-needed \
    lancedb=={LANCEDB_VERSION} tantivy sentence-transformers pyarrow tqdm
!pip uninstall -y triton 2>/dev/null; true

from google.colab import drive
drive.mount('/content/drive')

# CUDA-чек
import torch
assert torch.cuda.is_available(), (
    "CUDA недоступна. Runtime → Change runtime type → GPU."
)
print(f"torch={torch.__version__}, GPU={torch.cuda.get_device_name(0)}")


In [ ]:
# ==================== MODELS: 6 пресетов ====================
# Каждая запись — отдельная LanceDB-таблица. Можно отключить любую через enabled=False.
# `source` бывает трёх видов:
#   1) HF-идентификатор ("intfloat/multilingual-e5-base")
#   2) путь к .tar.gz на Drive (распакуется в /content/models/ автоматически)
#   3) путь к уже распакованной папке

MODELS = [
    {
        "name":         "rosberta-base",
        "table_name":   "rosberta-base-50k",
        "source":       "ai-forever/ru-en-RoSBERTa",
        "doc_prefix":   "",
        "query_prefix": "",
        "batch_size":   256,
        "enabled":      False,   # ← уже создана
    },
    {
        "name":         "rosberta-fine-tuned",
        "table_name":   "rosberta-fine-tuned-50k",
        "source":       "/content/drive/MyDrive/thesis/models/bi-encoder-rosberta-finetuned.tar.gz",
        "doc_prefix":   "",
        "query_prefix": "",
        "batch_size":   256,
        "enabled":      True,
    },
    {
        "name":         "e5-base-base",
        "table_name":   "e5-base-base-50k",
        "source":       "intfloat/multilingual-e5-base",
        "doc_prefix":   "passage: ",
        "query_prefix": "query: ",
        "batch_size":   384,
        "enabled":      False,   # ← уже создана
    },
    {
        "name":         "e5-base-fine-tuned",
        "table_name":   "e5-base-fine-tuned-50k",
        "source":       "/content/drive/MyDrive/thesis/models/bi-encoder-e5-finetuned.tar.gz",
        "doc_prefix":   "passage: ",
        "query_prefix": "query: ",
        "batch_size":   384,
        "enabled":      True,
    },
    {
        "name":         "bge-m3-base",
        "table_name":   "bge-m3-base-50k",
        "source":       "deepvk/USER-bge-m3",
        "doc_prefix":   "",
        "query_prefix": "",
        "batch_size":   128,
        "enabled":      True,
    },
    {
        "name":         "bge-m3-fine-tuned",
        "table_name":   "bge-m3-fine-tuned-50k",
        "source":       "/content/drive/MyDrive/thesis/models/bi-encoder-bge-m3-finetuned.tar.gz",
        "doc_prefix":   "",
        "query_prefix": "",
        "batch_size":   128,
        "enabled":      False,   # ← модель потеряна в Colab, пока не обучишь заново
    },
]

# --------- общие параметры (одинаковые для всех моделей) ---------
MAX_POSTS      = 50_000
# max_seq_length НЕ задаём — используем архитектурный дефолт модели (512 токенов)
MAX_SEQ_LEN    = None
RANDOM_SEED    = 42
DEVICE         = "cuda"
CLEAN_CATEGORY = True      # оставить только первую категорию до '|||'

enabled_count = sum(1 for m in MODELS if m['enabled'])
print(f"Включено моделей: {enabled_count} из {len(MODELS)}")
for m in MODELS:
    flag = '✓' if m['enabled'] else '·'
    print(f"  {flag} {m['name']:25s} → {m['table_name']}")


In [ ]:
# Пути на Colab + Drive
import os

DRIVE_BASE         = "/content/drive/MyDrive/thesis"
INPUT_JSONL        = os.path.join(DRIVE_BASE, "data/posts/ajtkulov/selected/selected500k_cleaned.jsonl")
DRIVE_OUTPUT_DIR   = os.path.join(DRIVE_BASE, "data/lancedb")
LOCAL_LANCEDB_PATH = "/content/lancedb_store"
LOCAL_MODELS_DIR   = "/content/models"

os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)
os.makedirs(LOCAL_MODELS_DIR, exist_ok=True)
os.makedirs(LOCAL_LANCEDB_PATH, exist_ok=True)

assert os.path.exists(INPUT_JSONL), f"Входной файл не найден: {INPUT_JSONL}"
print(f"Входной файл: {INPUT_JSONL}  ({os.path.getsize(INPUT_JSONL)/1e6:.1f} MB)")
print(f"Drive output:  {DRIVE_OUTPUT_DIR}")
print(f"Local LanceDB: {LOCAL_LANCEDB_PATH}")


In [ ]:
# ============================================================
# СЕМПЛИНГ ПОСТОВ — выполняется ОДИН РАЗ, до загрузки моделей.
# Эти 50 000 постов будут использованы для всех 6 таблиц.
# ============================================================
import json, random
from tqdm.auto import tqdm

all_posts = []
with open(INPUT_JSONL, 'r', encoding='utf-8') as f:
    for line in tqdm(f, desc="Чтение постов"):
        obj = json.loads(line)
        if not obj.get('text', '').strip():
            continue
        if CLEAN_CATEGORY:
            cat = obj.get('category', '')
            if '|||' in cat:
                obj['category'] = cat.split('|||')[0].strip()
        all_posts.append(obj)

print(f"Всего постов в файле: {len(all_posts):,}")

rng = random.Random(RANDOM_SEED)
posts = rng.sample(all_posts, MAX_POSTS)
del all_posts   # больше не нужны — освобождаем память

def make_post_id(post):
    return f"{post['channel']}::{post['id']}"

post_ids = [make_post_id(p) for p in posts]
print(f"Случайная выборка (seed={RANDOM_SEED}): {len(posts):,}")
print(f"Первый post_id:  {post_ids[0]}")
print(f"Последний post_id: {post_ids[-1]}")
print(f"Уникальных post_id: {len(set(post_ids)):,}  (должно быть = {MAX_POSTS:,})")

from collections import Counter
channels = set(p['channel'] for p in posts)
categories = Counter(p['category'] for p in posts)
print(f"Уникальных каналов:    {len(channels):,}")
print(f"Уникальных категорий:  {len(categories)}")


In [ ]:
# Универсальный распаковщик/резолвер моделей
import tarfile, time

def resolve_model(source: str) -> str:
    """Превращает source из MODELS в путь, который понимает SentenceTransformer."""
    # 1) Архив .tar.gz
    if source.endswith(".tar.gz") and os.path.exists(source):
        base_name = os.path.basename(source)[:-len(".tar.gz")]
        expected = os.path.join(LOCAL_MODELS_DIR, base_name)
        if os.path.exists(expected) and os.listdir(expected):
            print(f"  модель уже распакована: {expected}")
            return expected
        # Запоминаем, что было ДО распаковки
        before = set(os.listdir(LOCAL_MODELS_DIR)) if os.path.exists(LOCAL_MODELS_DIR) else set()
        print(f"  распаковка {source} → {LOCAL_MODELS_DIR} ...")
        t0 = time.time()
        with tarfile.open(source, "r:gz") as tar:
            tar.extractall(LOCAL_MODELS_DIR)
        print(f"    готово за {time.time()-t0:.0f}с")
        # Если ожидаемая папка появилась — отлично
        if os.path.isdir(expected):
            return expected
        # Иначе — ищем, какая новая папка появилась после распаковки
        after = set(os.listdir(LOCAL_MODELS_DIR))
        new_items = after - before
        new_dirs = [d for d in new_items if os.path.isdir(os.path.join(LOCAL_MODELS_DIR, d))]
        if len(new_dirs) == 1:
            actual = os.path.join(LOCAL_MODELS_DIR, new_dirs[0])
            # Переименовываем в ожидаемое имя, чтобы следующий архив не затёр
            print(f"  ⚠ в архиве папка '{new_dirs[0]}', переименовываем → '{base_name}'")
            os.rename(actual, expected)
            return expected
        # Если new_dirs == 0 — возможно папка уже была (от другого архива)
        # Ищем по содержимому: нужна папка с model.safetensors или config.json
        if len(new_dirs) == 0:
            for d in sorted(after):
                candidate = os.path.join(LOCAL_MODELS_DIR, d)
                if os.path.isdir(candidate) and d != base_name:
                    files_in = os.listdir(candidate)
                    if 'config.json' in files_in or 'model.safetensors' in files_in:
                        print(f"  ⚠ перезаписана папка '{d}', переименовываем → '{base_name}'")
                        os.rename(candidate, expected)
                        return expected
        raise FileNotFoundError(
            f"После распаковки {source} ожидалась папка {expected}, но её нет. "
            f"Новые элементы: {new_items}. Содержимое {LOCAL_MODELS_DIR}: {after}"
        )

    # 2) Существующая локальная папка
    if os.path.isdir(source):
        return source

    # 3) HF-идентификатор
    looks_like_hf = (
        "/" in source
        and not source.startswith(".")
        and not source.startswith("/")
        and not source.endswith(".tar.gz")
    )
    if looks_like_hf:
        return source

    raise FileNotFoundError(
        f"Не удалось разрешить source: {source!r}. "
        f"Это не существующий .tar.gz, не существующая папка и не похоже на HF-id."
    )


In [ ]:
# Функция, которая строит ОДНУ таблицу из общего списка `posts`
import gc, time
import pyarrow as pa
import lancedb
from sentence_transformers import SentenceTransformer

def build_table_for_model(cfg, posts, lancedb_path):
    """Загружает модель cfg, кодирует posts, пишет в таблицу cfg['table_name']."""
    name        = cfg['name']
    table_name  = cfg['table_name']
    source      = cfg['source']
    doc_prefix  = cfg['doc_prefix']
    batch_size  = cfg['batch_size']

    print(f"\n{'=' * 70}")
    print(f"  МОДЕЛЬ: {name}  →  таблица {table_name}")
    print(f"{'=' * 70}")

    model_path = resolve_model(source)
    print(f"  source: {source}")
    print(f"  path:   {model_path}")

    bi_encoder = SentenceTransformer(model_path, device=DEVICE)
    # max_seq_length выставляем ТОЛЬКО если MAX_SEQ_LEN не None —
    # иначе оставляем архитектурный дефолт модели (обычно 512)
    if MAX_SEQ_LEN is not None:
        bi_encoder.max_seq_length = MAX_SEQ_LEN
    dim = bi_encoder.get_sentence_embedding_dimension()
    print(f"  dim={dim}, max_seq_len={bi_encoder.max_seq_length}, batch_size={batch_size}")

    db = lancedb.connect(lancedb_path)

    schema = pa.schema([
        pa.field("vector",   pa.list_(pa.float32(), dim)),
        pa.field("text",     pa.utf8()),
        pa.field("channel",  pa.utf8()),
        pa.field("category", pa.utf8()),
        pa.field("post_id",  pa.utf8()),
        pa.field("link",     pa.utf8()),
        pa.field("date",     pa.utf8()),
        pa.field("views",    pa.utf8()),
    ])

    if table_name in db.table_names():
        db.drop_table(table_name)
        print(f"  старая таблица '{table_name}' удалена")

    table = db.create_table(table_name, schema=schema)

    total = len(posts)
    t_start = time.time()
    indexed = 0
    pbar = tqdm(total=total, desc=f"  {name}", unit="пост")

    for batch_start in range(0, total, batch_size):
        batch = posts[batch_start : batch_start + batch_size]
        texts = [p['text'] for p in batch]
        encoded_texts = [doc_prefix + t for t in texts] if doc_prefix else texts

        embeddings = bi_encoder.encode(
            encoded_texts,
            normalize_embeddings=True,
            show_progress_bar=False,
            batch_size=batch_size,
            device=DEVICE,
        )

        records = [{
            "vector":   embeddings[i].tolist(),
            "text":     batch[i]['text'],
            "channel":  batch[i]['channel'],
            "category": batch[i].get('category', ''),
            "post_id":  make_post_id(batch[i]),
            "link":     batch[i].get('link', ''),
            "date":     batch[i].get('date', ''),
            "views":    str(batch[i].get('views', '')),
        } for i in range(len(batch))]

        table.add(records)
        indexed += len(records)
        elapsed = time.time() - t_start
        speed = indexed / elapsed if elapsed > 0 else 0
        eta = (total - indexed) / speed if speed > 0 else 0
        pbar.update(len(records))
        pbar.set_postfix({"скор": f"{speed:.0f} п/с", "ETA": f"{eta/60:.1f}м"})
    pbar.close()

    print(f"  векторизация: {indexed:,} постов за {time.time()-t_start:.0f}с")
    print(f"  создание FTS-индекса...")
    table.create_fts_index("text", replace=True)
    print(f"  ✓ таблица {table_name}: {table.count_rows():,} строк, dim={dim}")

    # Освобождаем VRAM перед следующей моделью
    del bi_encoder
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return {
        "name":       name,
        "table_name": table_name,
        "rows":       table.count_rows(),
        "dim":        dim,
        "seconds":    round(time.time() - t_start, 1),
    }


In [ ]:
# Функция: упаковать готовую таблицу в .tar.gz и положить на Drive
import shutil

def archive_table_to_drive(table_name, lancedb_path, drive_output_dir):
    """Архивирует <table_name>.lance из lancedb_path и копирует на Drive."""
    table_dir = os.path.join(lancedb_path, f"{table_name}.lance")
    assert os.path.exists(table_dir), f"Таблица не найдена: {table_dir}"

    archive_name = f"{table_name}_lance.tar.gz"
    local_archive = f"/content/{archive_name}"
    drive_archive = os.path.join(drive_output_dir, archive_name)

    print(f"  архивирование {table_name}.lance → {archive_name} ...")
    !tar -czf {local_archive} -C {lancedb_path} {table_name}.lance
    size_mb = os.path.getsize(local_archive) / 1e6
    print(f"  размер архива: {size_mb:.1f} MB")

    print(f"  копирование на Drive: {drive_archive}")
    shutil.copy(local_archive, drive_archive)
    assert os.path.exists(drive_archive)
    print(f"  ✓ архив на Drive: {os.path.getsize(drive_archive)/1e6:.1f} MB")

    return drive_archive


In [ ]:
# ============================================================
# ГЛАВНЫЙ ЦИКЛ — строим таблицу за таблицей
# ============================================================
results = []
for cfg in MODELS:
    if not cfg['enabled']:
        print(f"\n[пропуск] {cfg['name']} — enabled=False")
        continue
    try:
        info = build_table_for_model(cfg, posts, LOCAL_LANCEDB_PATH)
        # Сразу архивируем и пушим на Drive — чтобы при дисконнекте Colab
        # уже посчитанные таблицы не пропали.
        archive_table_to_drive(cfg['table_name'], LOCAL_LANCEDB_PATH, DRIVE_OUTPUT_DIR)
        results.append(info)
    except Exception as e:
        print(f"  ✗ ОШИБКА для {cfg['name']}: {type(e).__name__}: {e}")
        results.append({"name": cfg['name'], "error": str(e)})

print("\n\n" + "=" * 70)
print("  СВОДКА")
print("=" * 70)
for r in results:
    if 'error' in r:
        print(f"  ✗ {r['name']}: {r['error']}")
    else:
        print(f"  ✓ {r['name']:25s} dim={r['dim']:5d}  rows={r['rows']:,}  {r['seconds']}с")


In [ ]:
# ============================================================
# ВЕРИФИКАЦИЯ: во всех успешно построенных таблицах post_id-set одинаков
# ============================================================
db = lancedb.connect(LOCAL_LANCEDB_PATH)
expected_ids = set(post_ids)
print(f"Эталонный набор post_id: {len(expected_ids):,}")

for r in results:
    if 'error' in r:
        continue
    t = db.open_table(r['table_name'])
    # to_lance() — нижележащий Lance-датасет, у которого проекция по столбцам
    # есть с самых ранних версий (в отличие от LanceTable.to_pandas(columns=...))
    table_ids = set(t.to_lance().to_table(columns=['post_id']).to_pandas()['post_id'].tolist())
    n_match = len(table_ids & expected_ids)
    n_extra = len(table_ids - expected_ids)
    n_miss  = len(expected_ids - table_ids)
    status  = '✓' if (n_match == len(expected_ids) and n_extra == 0) else '✗'
    print(f"  {status} {r['table_name']:30s} match={n_match:,}  extra={n_extra}  missing={n_miss}")


In [ ]:
# Размеры таблиц на диске
def dir_size_mb(path):
    total = 0
    for dp, _, fs in os.walk(path):
        for f in fs:
            total += os.path.getsize(os.path.join(dp, f))
    return total / (1024 * 1024)

for r in results:
    if 'error' in r:
        continue
    table_dir = os.path.join(LOCAL_LANCEDB_PATH, f"{r['table_name']}.lance")
    size = dir_size_mb(table_dir)
    print(f"  {r['table_name']:30s} {size:7.1f} MB  ({size*1024/r['rows']:.1f} KB/запись)")


In [ ]:
# Санити-проверка: пробуем один и тот же запрос на каждой таблице
test_query = "Кроссовки Nike Air Max мужские для бега, размер 42, чёрные"
print(f"Запрос: {test_query}\n")

for cfg in MODELS:
    if not cfg['enabled']:
        continue
    if not any(r.get('table_name') == cfg['table_name'] and 'error' not in r for r in results):
        continue
    print(f"--- {cfg['name']} ---")
    t = db.open_table(cfg['table_name'])
    # Используем BM25 — не нужно загружать модель ещё раз
    rows = (t.search(test_query, query_type='fts')
            .limit(3).select(['text','channel','category']).to_list())
    for i, r in enumerate(rows, 1):
        print(f"  {i}. [{r['category']}] @{r['channel']}")
        print(f"     {r['text'][:120]}...")
    print()


## После прогона

На Drive в `thesis/data/lancedb/` лежат 6 архивов:

```
rosberta-base-50k_lance.tar.gz
rosberta-fine-tuned-50k_lance.tar.gz
e5-base-base-50k_lance.tar.gz
e5-base-fine-tuned-50k_lance.tar.gz
bge-m3-base-50k_lance.tar.gz
bge-m3-fine-tuned-50k_lance.tar.gz
```

Скачиваешь, распаковываешь каждый в `thesis/lancedb_store/`:

```bash
cd thesis/
for f in /path/to/*-50k_lance.tar.gz; do
    tar -xzf "$f" -C lancedb_store/
done
```

## Что дальше

Все 6 таблиц содержат **один и тот же набор** `post_id` (проверка в
верификационной ячейке выше). Бенчмарк теперь может честно сравнивать модели:
разница в качестве поиска объясняется только эмбеддингами, а не выборкой данных.
